In [ ]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_silver_users"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Users_Inventory" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_Users" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 3, Finished, Available, Finished)

🔧 Initializing ntk_silver_users...


🚀 Starting ntk_silver_users


### Paths for file reading and file saving

In [ ]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 4, Finished, Available, Finished)

### csv file reading form bronze layer

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lit, current_timestamp, to_timestamp, row_number, coalesce, when,
    regexp_replace, upper, lower, isnan, isnull, concat_ws, desc
)
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.window import Window

# Step 1: Initialize Spark Session with optimized configurations
spark = SparkSession.builder.appName("BronzeToSilver_ListsLibrary").getOrCreate()

## path
today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 
bronze_path = f"{source_path}/{year}/{month}/{day}/Users_Inventory.csv"

# Step 2: Date-based path setup
current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"

silver_path = f"{target_path}/{year}/{month}/{day}/Dim_User.parquet"

# Step 3: Read CSV with schema inference

df_raw_full = spark.read.option("header", True).csv(bronze_path)
df_raw = df_raw_full.where(col("PrincipalType") == "User")
df_raw.show(1)

StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 5, Finished, Available, Finished)

+------+----------------+--------------------+-------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------+--------------------+--------------+------------+--------+--------+---------------+--------------+-----------+-----------------------+-----------+------------+--------------+---------------+------------------+--------------------+--------------------+--------------------+------------------+-----------+--------------+------------+-------+----------+----------------+--------------------+--------------------+---------------------+--------------------+----------------+----------+-------------+--------------+--------------+------------------+-----------------+----------------+----------+----------------+--------------+---------------+--------------------+-------------+----------+------------+
|UserId|        UserGuid|           LoginName|PrincipalType|   UserPrincipalName|              SiteId|     SiteName|             S

## Add metadata, Convert all columns to string, Trim string columns,Boolean normalization, Date normalization and Duplication removal


In [ ]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, lit
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import StringType

# Step 1: Add metadata columns
df_raw = df_raw.withColumn("SnapshotDate", current_timestamp()) \
               .withColumn("ProcessedDate", current_timestamp()) \
               .withColumn("DataSource", lit("SharePoint_Users_Inventory"))


# Step 2: Convert all columns to string (except metadata)
for field in df_raw.schema.fields:
    if field.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]:
        df_raw = df_raw.withColumn(field.name, col(field.name).cast(StringType()))

# Step 3: Trim string columns
string_cols = [f.name for f in df_raw.schema.fields if isinstance(f.dataType, StringType) and f.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]]
for col_name in string_cols:
    df_raw = df_raw.withColumn(col_name, trim(col(col_name)))

# Step 4: Standardize nulls
null_values = ["", "NULL", "null", "N/A", "n/a"]
for null_val in null_values:
    df_raw = df_raw.replace(null_val, None)

# Step 5: Boolean normalization
bool_cols = ["IsSystemAccount","IsExternalUser","IsGuestUser","IsShareByEmailGuestUser","IsSiteAdmin",
             "IsHiddenInUI","HasDirectPermissions","IsGroupMember","ReviewRequired","IsFromSubsite","SafeModeUsed"]
for col_name in bool_cols:
    if col_name in df_raw.columns:
        df_raw = df_raw.withColumn(col_name, 
                                   when(upper(col(col_name)).isin(["TRUE","1","YES","Y"]), True)
                                   .when(upper(col(col_name)).isin(["FALSE","0","NO","N"]), False)
                                   .otherwise(None))

# Step 6: Numeric normalization
num_cols = ["UserId", "DirectPermissionCount", "GroupCount"]
for col_name in num_cols:
    if col_name in df_raw.columns:
        df_raw = df_raw.withColumn(col_name,
                                   when(col(col_name).rlike("^[0-9]+$"), col(col_name).cast(IntegerType()))
                                   .when(col(col_name).rlike("^[0-9]*\\.?[0-9]+[Ee][+-]?[0-9]+$"), col(col_name).cast(DoubleType()).cast(IntegerType()))
                                   .otherwise(None))

# Step 7: Date normalization
date_cols = ["LastAccessDate","InvitationDate","RecordedDateTime"]
for col_name in date_cols:
    if col_name in df_raw.columns:
        df_raw = df_raw.withColumn(col_name, to_timestamp(col(col_name), "M/d/yyyy h:mm:ss a"))

# Step 8: Filter out records with null UserId
if "UserId" in df_raw.columns:
    df_raw = df_raw.filter(col("UserId").isNotNull())

# Step 9: Remove Deduplicate based on UserId (most recent record)
if "UserId" in df_raw.columns:
    sort_col = "RecordedDateTime" if "RecordedDateTime" in df_raw.columns else "UserId"
    window_spec = Window.partitionBy("UserId").orderBy(desc(sort_col))
    df_final = df_raw.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num")==1).drop("row_num")
else:
    df_final = df_raw.dropDuplicates()
# Step 6: Handle UserGuid logic (if UserGuid column exists)

if "UserGuid" in df_final.columns:
    df_cleaned = df_final.withColumn(
        "UserGuid_Clean",
        when(
            (col("UserType") == "External") &
            (col("UserGuid").rlike("^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")),
            col("UserGuid")
        ).when(
            (F.col("IsExternalUser") == False) &
            (F.col("UserGuid").rlike("^[0-9]+[a-z0-9]*$")),
            col("UserGuid")
        ).otherwise(lit(None).cast(StringType()))
    )

# Step 10: Keep all columns (including empty ones)
# No need to remove columns; just keep all columns as is
# You can skip this part since no removal is happening

# Write to Silver layer (optional, currently commented out)
# df_final.write.mode("overwrite").option("compression", "snappy").parquet(silver_path)


StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 6, Finished, Available, Finished)

In [ ]:
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import StringType

from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
# Step 4: Add metadata columns
df_raw = df_raw.withColumn("SnapshotDate", current_timestamp()) \
              .withColumn("ProcessedDate", current_timestamp()) \
              .withColumn("DataSource", lit("SharePoint_Users_Inventory"))

# Convert all columns to string type for consistent processing (except metadata columns)
df_string_converted = df_raw
for field in df_raw.schema.fields:
    if field.name not in ["SnapshotDate", "ProcessedDate"]:  # Keep timestamp columns
        df_string_converted = df_string_converted.withColumn(field.name, col(field.name).cast(StringType()))

# Trim whitespace from all string columns
string_cols = [field.name for field in df_string_converted.schema.fields 
               if field.dataType == StringType() and field.name not in ["SnapshotDate", "ProcessedDate", "DataSource"]]
for col_name in string_cols:
    df_string_converted = df_string_converted.withColumn(col_name, trim(col(col_name)))

# Replace various representations of null/empty values
null_replacements = ["", "NULL", "null", "N/A", "n/a"]
df_cleaned = df_string_converted
for null_val in null_replacements:
    df_cleaned = df_cleaned.replace(null_val, None)

# Step 6: Handle UserGuid logic (if UserGuid column exists)

if "UserGuid" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        "UserGuid_Clean",
        when(
            (col("UserType") == "External") &
            (col("UserGuid").rlike("^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")),
            col("UserGuid")
        ).when(
            (F.col("IsExternalUser") == False) &
            (F.col("UserGuid").rlike("^[0-9]+[a-z0-9]*$")),
            col("UserGuid")
        ).otherwise(lit(None).cast(StringType()))
    )

StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 7, Finished, Available, Finished)

In [ ]:


# Update UserType for Internal users
df_cleaned = df_cleaned.withColumn(
    "UserType",
    when(
        (F.col("IsExternalUser") == False) &
        (F.col("UserGuid").rlike("^[0-9]+[a-z0-9]*$")),
        lit("Internal")
    ).otherwise(col("UserType"))
)

from pyspark.sql.functions import format_string

df_cleaned = df_cleaned.withColumn("ClaimsIdentifier_Full", format_string("%.0f", col("ClaimsIdentifier")))

# Step 7: Create standardized boolean columns (check if columns exist)
potential_boolean_columns = [
    "IsSystemAccount", "IsExternalUser", "IsGuestUser", 
    "IsShareByEmailGuestUser", "IsSiteAdmin", "IsHiddenInUI",
    "HasDirectPermissions", "IsGroupMember", "ReviewRequired",
    "IsFromSubsite", "SafeModeUsed"
]

actual_boolean_columns = [col_name for col_name in potential_boolean_columns if col_name in df_cleaned.columns]

for bool_col in actual_boolean_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{bool_col}_Clean",
        when(upper(col(bool_col)).isin(["TRUE", "1", "YES", "Y"]), True)
        .when(upper(col(bool_col)).isin(["FALSE", "0", "NO", "N"]), False)
        .otherwise(None)
    )

# Step 8: Handle date columns with proper parsing (check if columns exist)
potential_date_columns = ["LastAccessDate", "InvitationDate", "RecordedDateTime"]
actual_date_columns = [col_name for col_name in potential_date_columns if col_name in df_cleaned.columns]

for date_col in actual_date_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{date_col}_Clean",
        when(col(date_col).isNotNull() & (col(date_col) != ""), 
             to_timestamp(col(date_col), "M/d/yyyy H:mm"))
        .otherwise(None)
    )

# Step 9: Handle numeric columns (check if columns exist)
potential_numeric_columns = ["UserId", "DirectPermissionCount", "GroupCount"]
actual_numeric_columns = [col_name for col_name in potential_numeric_columns if col_name in df_cleaned.columns]

for num_col in actual_numeric_columns:
    df_cleaned = df_cleaned.withColumn(
        f"{num_col}_Clean",
        when(col(num_col).rlike("^[0-9]+$"), col(num_col).cast(IntegerType()))
        .when(col(num_col).rlike("^[0-9]*\\.?[0-9]+[Ee][+-]?[0-9]+$"), col(num_col).cast(DoubleType()).cast(IntegerType()))
        .otherwise(None)
    )

# Step 11: Data filtering
df_filtered = df_cleaned.filter(col("UserGuid_Clean").isNotNull())

# Step 12: Deduplication strategy
# Remove duplicates based on UserGuid, keeping the most recent record
dedup_columns = []
if "RecordedDateTime_Clean" in df_filtered.columns:
    dedup_columns.append(desc("RecordedDateTime_Clean"))
elif "RecordedDateTime" in df_filtered.columns:
    dedup_columns.append(desc("RecordedDateTime"))
elif "UserId_Clean" in df_filtered.columns:
    dedup_columns.append(desc("UserId_Clean"))
elif "UserId" in df_filtered.columns:
    dedup_columns.append(desc("UserId"))
else:
    # If no sorting column available, just use the first occurrence
    dedup_columns.append(desc("UserGuid_Clean"))

window_spec = Window.partitionBy("UserGuid_Clean").orderBy(*dedup_columns)

df_deduplicated = df_filtered.withColumn("row_num", row_number().over(window_spec)) \
                            .filter(col("row_num") == 1) \
                            .drop("row_num")


StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 8, Finished, Available, Finished)

In [ ]:
final_columns_mapping = {}

for source_col, target_col in final_columns_mapping.items():
    select_expressions.append(col(source_col).alias(target_col))
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, sha2, concat_ws, when, coalesce
from pyspark.sql.functions import col, lit, current_timestamp, sha2, concat_ws
from datetime import datetime
import os
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, lit, current_timestamp, sha2, concat_ws
# Create dynamic mapping based on available columns

# Step 13: Dynamic column mapping - map existing columns to target schema
# Ensure UserPrincipleName is included
if "UserPrincipalName" in df_deduplicated.columns:
    final_columns_mapping["UserPrincipalName"] = "UserPrincipalName"

available_columns = df_deduplicated.columns
# print(f"Available columns after processing: {available_columns}")
# df_deduplicated.write.mode("overwrite").parquet(silver_path)


# Core columns mapping
column_mappings = {
    "UserGuid_Clean": "UserGuid",
    "DisplayName": "DisplayName", 
    "Email": "Email",
    "UserPrincipalName": "UserPrincipalName",
    "PrincipalType": "PrincipalType",
    "UserType": "UserType",
    "ComplianceStatus": "ComplianceStatus",
    "LoginName": "LoginName",
    "ClaimsIdentifier": "ClaimsIdentifier",
    "ClaimsIssuer": "ClaimsIssuer",
    "DirectPermissions": "DirectPermissions",
    "GroupMemberships": "GroupMemberships",
    "SubsiteUrl": "SubsiteUrl",
    "SiteId": "SiteId",
    "SiteName": "SiteName",
    "SiteUrl": "SiteUrl",
    "WebId": "WebId",
    "Title": "Title",
    "SnapshotDate": "SnapshotDate",
    "ProcessedDate": "ProcessedDate",
    "DataSource": "DataSource"
}

# Add cleaned columns if they exist
for bool_col in actual_boolean_columns:
    column_mappings[f"{bool_col}_Clean"] = bool_col

for date_col in actual_date_columns:
    column_mappings[f"{date_col}_Clean"] = date_col

for num_col in actual_numeric_columns:
    column_mappings[f"{num_col}_Clean"] = num_col

# Only include mappings where source column exists
for source_col, target_col in column_mappings.items():
    if source_col in available_columns:
        final_columns_mapping[source_col] = target_col

print(f"Final column mapping: {final_columns_mapping}")

# Step 14: Final column selection and renaming
select_expressions = []
# df_final = df_deduplicated.select(*select_expressions)

# Step 17: Data validation checks
validation_checks = {
    "total_records": df_final.count()
}



StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 9, Finished, Available, Finished)

Final column mapping: {'UserPrincipalName': 'UserPrincipalName', 'UserGuid_Clean': 'UserGuid', 'DisplayName': 'DisplayName', 'Email': 'Email', 'PrincipalType': 'PrincipalType', 'UserType': 'UserType', 'ComplianceStatus': 'ComplianceStatus', 'LoginName': 'LoginName', 'ClaimsIdentifier': 'ClaimsIdentifier', 'ClaimsIssuer': 'ClaimsIssuer', 'DirectPermissions': 'DirectPermissions', 'GroupMemberships': 'GroupMemberships', 'SubsiteUrl': 'SubsiteUrl', 'SiteId': 'SiteId', 'SiteName': 'SiteName', 'SiteUrl': 'SiteUrl', 'WebId': 'WebId', 'Title': 'Title', 'SnapshotDate': 'SnapshotDate', 'ProcessedDate': 'ProcessedDate', 'DataSource': 'DataSource', 'IsSystemAccount_Clean': 'IsSystemAccount', 'IsExternalUser_Clean': 'IsExternalUser', 'IsGuestUser_Clean': 'IsGuestUser', 'IsShareByEmailGuestUser_Clean': 'IsShareByEmailGuestUser', 'IsSiteAdmin_Clean': 'IsSiteAdmin', 'IsHiddenInUI_Clean': 'IsHiddenInUI', 'HasDirectPermissions_Clean': 'HasDirectPermissions', 'IsGroupMember_Clean': 'IsGroupMember', 'Revi

In [ ]:
# Only run validation if columns exist
if "UserGuid" in df_final.columns:
    validation_checks["null_userguids"] = df_final.filter(col("UserGuid").isNull()).count()
    validation_checks["duplicate_userguids"] = df_final.groupBy("UserGuid").count().filter(col("count") > 1).count()

if "Email" in df_final.columns:
    validation_checks["invalid_emails"] = df_final.filter(
        col("Email").isNotNull() & 
        ~col("Email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$")
    ).count()


StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 10, Finished, Available, Finished)

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

# Step 1: Create GUID column based on conditions
df_finaltemp = df_final.withColumn(
    "UserGUIDCol",
    when(
        (col("UserPrincipalName").isNotNull()) & 
        (col("UserPrincipalName") != "") & 
        (col("UserPrincipalName") != " "),
        col("UserPrincipalName")
    ).otherwise(
        when(
            (col("DisplayName").isNotNull()) & 
            (col("DisplayName") != "") & 
            (col("DisplayName") != " "),
            col("DisplayName")
        ).otherwise(lit(None))
    )
)
    
# Step 2: Convert GUID to SHA256 hash
df_final = df_finaltemp.withColumn(
    "User_GUIDPK",
    when(
        col("UserGUIDCol").isNotNull(),
        sha2(col("UserGUIDCol"), 256)
    ).otherwise(lit(None))
)

# Step 3: Write to Silver layer with partitioning
try:
    df_final.write.mode("overwrite").parquet(silver_path)
    
    print(f"Successfully wrote data to Silver layer: {silver_path}")
    
except Exception as e:
    print(f"Error writing to Silver layer: {str(e)}")
    raise

print("=== Data Validation Results ===")
for check_name, count in validation_checks.items():
    print(f"{check_name}: {count}")
    if count > 0 and check_name != "total_records":
        print(f"WARNING: Found {count} records with {check_name}")

# # Show sample of final data
# print("=== Sample of Final Data ===")
df_final.show(2, truncate=False)

# # Show final schema
# print("=== Final Schema ===")
# df_final.printSchema()

# Stop Spark session
print("ETL process completed successfully!")



StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 11, Finished, Available, Finished)

Successfully wrote data to Silver layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_User.parquet
=== Data Validation Results ===
total_records: 21
null_userguids: 0
duplicate_userguids: 0
invalid_emails: 0
+------+------------------------------------------------------------+-----------------------------------------------+-------------+-----------------------------+------------------------------------+-------------+------------------------------------------------------------+------------------------------------+-------------------+-----------------------------+-------------------+---------+--------+--------+---------------+--------------+-----------+-----------------------+-----------+------------+--------------+---------------+------------------+----------+-------------------------+----------+------------------+-----------+--------------+------------+-------+----------+------------------------------

In [ ]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = df_raw.count()
    rows_written = df_final.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 775b40d9-3168-43ab-8eb4-7679b2435c86, 12, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_silver_users...


✅ ntk_silver_users completed successfully (364s)
🎉 ntk_silver_users pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 21 → 21
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_silver_users:


+-----+-------+--------------------+--------------------+---------------+--------+-----------+
|LogID| Status|           StartTime|             EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+-------+--------------------+--------------------+---------------+--------+-----------+
|  119|RUNNING|2025-10-15 05:34:...|                NULL|           NULL|    NULL|       NULL|
|  121|SUCCESS|2025-10-15 05:34:...|2025-10-15 05:40:...|            364|      21|         21|
|  100|RUNNING|2025-10-14 10:45:...|                NULL|           NULL|    NULL|       NULL|
+-----+-------+--------------------+--------------------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_silver_users logging completed!
